In [1]:
import os
import json
import nltk
from nltk.stem import PorterStemmer
import re
from dotenv import load_dotenv
from google import genai
print(genai.__file__)


# Download necessary NLTK data
nltk.download('punkt', quiet=True)

# --- JSON data for limited chatbot responses ---
longevity_data = {
    "diet": {
        "keywords": ["diet", "food", "eat", "nutrition"],
        "response": "A Mediterranean-style diet, rich in vegetables, fruits, whole grains, healthy fats (like olive oil), and lean proteins, is often associated with improved longevity. Focus on nutrient-dense, unprocessed foods and mindful eating habits."
    },
    "exercise": {
        "keywords": ["exercise", "workout", "physical activity", "move"],
        "response": "Regular physical activity, combining aerobic exercises (like walking, running), strength training (weights, bodyweight), and flexibility/balance work (yoga), is vital for cardiovascular health, muscle maintenance, and bone density, all contributing to a longer, healthier life."
    },
    "sleep": {
        "keywords": ["sleep", "rest", "insomnia"],
        "response": "Adequate, high-quality sleep (7-9 hours for most adults) is crucial. It allows the body to repair, consolidate memories, and regulate hormones. Establish a consistent sleep schedule and create a relaxing bedtime routine."
    },
    "stress": {
        "keywords": ["stress", "anxiety", "mindfulness"],
        "response": "Chronic stress can accelerate aging. Techniques like mindfulness meditation, deep breathing exercises, spending time in nature, and maintaining strong social connections are effective strategies for stress management and improving longevity."
    },
    "supplements": {
        "keywords": ["supplements", "vitamins", "pills"],
        "response": "While some supplements like Vitamin D, Omega-3 fatty acids, or certain probiotics might be beneficial for specific deficiencies or conditions, it's generally best to consult a healthcare professional before starting any supplement regimen. A balanced diet should be your primary source of nutrients."
    },
    "hydration": {
        "keywords": ["water", "hydrate", "drinks"],
        "response": "Staying well-hydrated is fundamental for all bodily functions, including metabolism, nutrient transport, and detoxification. Aim to drink plenty of water throughout the day, adjusting based on activity level and climate."
    },
    "social connections": {
        "keywords": ["friends", "family", "social", "community"],
        "response": "Strong social ties and a sense of community are increasingly recognized as critical factors for longevity. Engaging with others, fostering meaningful relationships, and participating in group activities can reduce stress and improve mental well-being."
    },
    "brain health": {
        "keywords": ["brain", "memory", "cognitive"],
        "response": "Keeping your brain active through lifelong learning, puzzles, reading, and new experiences can support cognitive longevity. A healthy diet, regular exercise, and good sleep also play a significant role in brain health."
    },
    "genetics": {
        "keywords": ["genetics", "DNA", "heredity"],
        "response": "While genetics play a role in longevity, lifestyle factors often have a greater impact. Even with a predisposition, healthy habits can significantly influence your healthspan and lifespan."
    },
    "purpose": {
        "keywords": ["purpose", "meaning", "goals"],
        "response": "Having a sense of purpose or meaning in life has been linked to increased longevity and well-being. This could involve hobbies, volunteering, career, or personal goals that bring you fulfillment."
    },
    "default": {
        "response": "I'm currently operating in a limited mode. To get more advanced responses, please configure your Google Generative AI API key. Meanwhile, you can ask about **diet**, **exercise**, **sleep**, **stress**, **hydration**, or **social connections**."
    }
}

# Save JSON file
file_name = "longevity_responses.json"
with open(file_name, "w") as f:
    json.dump(longevity_data, f, indent=4)

print(f"Generated '{file_name}' with expanded longevity responses.")


/usr/local/lib/python3.12/dist-packages/google/genai/__init__.py
Generated 'longevity_responses.json' with expanded longevity responses.


In [2]:
# --- Select mode and load API key safely ---
from google.colab import files
from getpass import getpass
from dotenv import load_dotenv

GOOGLE_API_KEY = None  # default

print("Please select an option for running the chatbot:")
print("1: Upload a .env file with your API key")
print("2: Enter API key manually")
print("3: Run in limited mode (no API key)")

choice = input("Enter 1, 2, or 3: ").strip()

if choice == "1":
    uploaded_files = files.upload()
    # Handle Colab renaming like .env 1, .env 2, etc.
    for uploaded_filename in uploaded_files.keys():
        print(f"✅ Uploaded file detected: {uploaded_filename}")
        # Load the actual uploaded file
        load_dotenv(uploaded_filename)
        # Optional: rename to '.env' for future runs
        target_name = ".env"
        try:
            if uploaded_filename != target_name:
                os.rename(uploaded_filename, target_name)
                print(f"File renamed to '{target_name}' for future runs.")
        except FileExistsError:
            print(f"Note: '{target_name}' already exists. Keeping filename as {uploaded_filename}")

    # Read key from environment after loading
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
    if GOOGLE_API_KEY:
        print("✅ GOOGLE_API_KEY loaded successfully from uploaded .env file.")
    else:
        print("⚠️ Could not find GOOGLE_API_KEY in the uploaded file. Running limited mode.")

elif choice == "2":
    GOOGLE_API_KEY = getpass("Enter your Google Generative AI API Key (input hidden): ")
    if GOOGLE_API_KEY:
        os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
        print("✅ API key stored for this session.")
    else:
        print("⚠️ No API key entered. Running limited mode.")

elif choice == "3":
    print("⚠️ Running in limited mode (no API key).")

else:
    print("⚠️ Invalid choice. Defaulting to limited mode.")

# Ensure environment variable is always set for the chatbot
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY if GOOGLE_API_KEY else ""


Please select an option for running the chatbot:
1: Upload a .env file with your API key
2: Enter API key manually
3: Run in limited mode (no API key)
Enter 1, 2, or 3: 2
Enter your Google Generative AI API Key (input hidden): ··········
✅ API key stored for this session.


In [3]:
# --- Stemmer setup for keyword matching in limited mode ---
nltk.download('punkt_tab', quiet=True)
stemmer = PorterStemmer()

def stem_text(text):
    """Stem user input for keyword matching."""
    tokens = nltk.word_tokenize(text.lower())
    return ' '.join([stemmer.stem(t) for t in tokens if t.isalpha()])

# Pre-stem keywords from JSON for efficiency
with open("longevity_responses.json", "r") as f:
    limited_responses_data = json.load(f)

stemmed_limited_responses_data = {}
for topic, data in limited_responses_data.items():
    stemmed_keywords = [stem_text(k) for k in data.get("keywords", [])]
    stemmed_limited_responses_data[topic] = {
        "stemmed_keywords": stemmed_keywords,
        "response": data["response"]
    }
if "default" in limited_responses_data:
    stemmed_limited_responses_data["default"] = limited_responses_data["default"]


In [4]:
# --- Cell 4: Longevity Chatbot ---
import sys
sys.path.append("/content")  # add folder where llm_client.py is located

from llm_client import generate_llm_response  # use the new client
# if stem_text is in cell 3, you can just use it directly

def longevity_chatbot():
    print("Initializing Longevity Chatbot...")

    llm_mode = True if os.environ.get("GOOGLE_API_KEY") else False

    if llm_mode:
        print("\n✅ 🧠 Longevity Chatbot — LLM Mode Activated")
        print("You're now chatting with an AI assistant trained to provide insights about longevity, health, and wellness.")
        print("Type 'exit' or 'quit' anytime to end the chat.")
    else:
        topics = ", ".join([f"**{t}**" for t in limited_responses_data if t != "default"])
        print(f"\n⚠️ Limited mode active. You can ask about: {topics}")

    while True:
        user_input = input("\nType your question: ").strip()
        if user_input.lower() in ['quit', 'exit']:
            print("Chatbot: Goodbye!")
            break

        # --- LLM mode ---
        if llm_mode:
            try:
                response = generate_llm_response(user_input)
                print(f"Chatbot: {response}")
                continue  # skip limited mode
            except Exception as e:
                print(f"⚠️ LLM API call failed: {e}")
                print("Falling back to limited mode for this turn.")
                llm_mode = False

        # --- Limited mode fallback ---
        user_input_stem = stem_text(user_input)
        found = False
        for topic, data in stemmed_limited_responses_data.items():
            if topic != "default" and any(k in user_input_stem for k in data["stemmed_keywords"]):
                print(f"Chatbot (Limited): {data['response']}")
                found = True
                break
        if not found:
            print(f"Chatbot (Limited): {stemmed_limited_responses_data['default']['response']}")

# Launch chatbot
longevity_chatbot()


Initializing Longevity Chatbot...

✅ 🧠 Longevity Chatbot — LLM Mode Activated
You're now chatting with an AI assistant trained to provide insights about longevity, health, and wellness.
Type 'exit' or 'quit' anytime to end the chat.

Type your question: exercising
Chatbot: That's a great topic! Exercising is one of the best things you can do for your overall health and well-being.

It encompasses a wide range of activities, all designed to improve physical fitness and health.

Here's a quick rundown of why exercising is so important and some common aspects:

**Why Exercise? (Benefits)**

*   **Physical Health:**
    *   Strengthens your heart and improves cardiovascular health.
    *   Helps manage weight and reduces the risk of obesity.
    *   Builds and maintains strong muscles and bones.
    *   Improves flexibility, balance, and coordination.
    *   Boosts your immune system.
    *   Increases energy levels and stamina.
    *   Improves sleep quality.
    *   Reduces the risk of 